# Hybrid expression checks: chondrocyte vs osteochondral, and gorilla-specific genes

Chondrocyte vs osteochondral-progenitor expression similarity (Pearson's r = 0.745), and 3,766 genes whose expression differs in gorilla relative to both humans and chimpanzees.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Paths (EDIT): inputs read / outputs written by this notebook ---
NCBI_ANNOT_TSV = '/home/labs/davidgo/nadavmi/usefull/Human.GRCh38.p13.annot.tsv'  # NCBI GRCh38.p13 gene annotation (GeneID -> Symbol)
ARTICULAR_CARTILAGE_TPM_TSV = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/GSE114007_human_control_and_osteoarithritis_cartilage/GSE114007_norm_counts_TPM_GRCh38.p13_NCBI.tsv'  # GSE114007 articular cartilage TPM
HYBRID_POLARIZATION_TSV = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Hybrids/human_chimp/ExpLBM/outputs_05Jan2026_humanMPRA_draft1/ExpLBM_polarization_results.tsv'  # human-chimp hybrid ExpLBM polarization results (draft outputs)
YAMADA_RAW_XLSX = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/GSE165620_Yamada_differentiation_protocol/GSE165620_hiPSC_RNAseq_raw_data.xlsx'  # GSE165620 Yamada raw RNA-seq (column names)
YAMADA_TPM_TSV = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/GSE165620_Yamada_differentiation_protocol/GSE165620_norm_counts_TPM_GRCh38.p13_NCBI.tsv'  # GSE165620 Yamada normalized TPM
NCBI_ANNOT_TSV_OLD = '/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/NCBI/hg38/old/Human.GRCh38.p13.annot.tsv'  # NCBI GRCh38.p13 annotation (old GenomeAnnotation copy)
HYBRIDS_HG_POLARIZATION_TSV = '/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/hybrids_TPM/ExpLBM_HG_polarization_results.tsv'  # human-gorilla hybrid ExpLBM polarization results
HYBRIDS_HC_POLARIZATION_TSV = '/home/labs/davidgo/Collaboration/humanMPRA/paper/extended_datasets/hybrids_TPM/ExpLBM_polarization_results.tsv'  # human-chimp hybrid ExpLBM polarization results (extended datasets)
output_path = '.'  # where plots/tables are written


## Comparing the activity of the Osteochondral progenitor cells to the chondroycts

### Load the articular cartilage TPM

In [ ]:
gene_annotation_table = pd.read_csv(NCBI_ANNOT_TSV, 
                     header=0,sep = '\t',usecols=[0,1])

articular_cartilage_TPM = pd.read_csv(ARTICULAR_CARTILAGE_TPM_TSV, 
                     header=0,sep = '\t')

articular_cartilage_TPM = articular_cartilage_TPM.iloc[:,0:9]
articular_cartilage_TPM['articular_cartilage_mean'] = articular_cartilage_TPM.iloc[:, 1:].mean(axis=1)
articular_cartilage_TPM = articular_cartilage_TPM[['GeneID','articular_cartilage_mean']]


articular_cartilage_TPM = pd.merge(articular_cartilage_TPM, gene_annotation_table, on='GeneID', how='outer') 
articular_cartilage_TPM = articular_cartilage_TPM.set_index('Symbol')
articular_cartilage_TPM = articular_cartilage_TPM[['articular_cartilage_mean']]
articular_cartilage_TPM = articular_cartilage_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates


### Load the hybrids TPM

In [ ]:
hybrid_data = pd.read_csv(HYBRID_POLARIZATION_TSV, 
                     sep = '\t',
                     header=0)
hybrid_data = hybrid_data[hybrid_data['Gene'].notna()]
hybrid_data['ASE_full'] = hybrid_data['ExpLBM_gene_ase_type']+'_'+hybrid_data['derived']

In [ ]:
# Compare TPM in chondrocytes vs hybrids
# Merge data by gene name
comparison_df = articular_cartilage_TPM.reset_index().rename(columns={'Symbol': 'Gene', 'articular_cartilage_mean': 'chond_TPM'})
comparison_df = comparison_df.merge(
    hybrid_data[['Gene', 'ExpLBM_TPM_human_allele']].drop_duplicates(subset=['Gene'], keep='first'),
    on='Gene',
    how='inner'
)
comparison_df = comparison_df.dropna(subset=['chond_TPM', 'ExpLBM_TPM_human_allele'])

# Create hexbin plot comparing TPM values
fig, ax = plt.subplots(figsize=(9, 7))

# Add pseudocount of 0.1 to avoid log(0) issues
x_data = comparison_df['chond_TPM'].values + 0.1
y_data = comparison_df['ExpLBM_TPM_human_allele'].values + 0.1

# Create hexbin plot with log scale
hb = ax.hexbin(x_data, y_data, gridsize=100, xscale='log', yscale='log', linewidths=0,
               cmap=custom_cmap, mincnt=1, vmax=25)

# Add diagonal reference line
min_val = min(x_data.min(), y_data.min())
max_val = max(x_data.max(), y_data.max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, alpha=0.5, label='Perfect agreement')

# Calculate correlation
pearson_r, pearson_p = pearsonr(np.log10(x_data), np.log10(y_data))
spearman_r, spearman_p = spearmanr(x_data, y_data)

ax.set_xlabel('Chondrocyte TPM (log10 scale)', fontsize=FONT_SIZE_small)
ax.set_ylabel('Hybrid TPM human allele (log10 scale)', fontsize=FONT_SIZE_small)
#ax.set_title(f'TPM Comparison: Chondrocytes vs Hybrids\n(n={len(comparison_df)} genes)', fontsize=FONT_SIZE_small)
ax.legend(frameon=False, fontsize=FONT_SIZE_small-2, loc='lower right')
ax.grid(True, alpha=0.3)

# Add colorbar
cb = plt.colorbar(hb, ax=ax)
cb.set_label('Count', fontsize=FONT_SIZE_small-2)

# Add correlation text
textstr = f'Pearson r = {pearson_r:.3f} (p={pearson_p:.2e})\nSpearman ρ = {spearman_r:.3f} (p={spearman_p:.2e})'
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=FONT_SIZE_small-2,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(output_path + 'chondrocytes_vs_hybrids_TPM_comparison.pdf', bbox_inches='tight', dpi=300)
plt.savefig(output_path + 'chondrocytes_vs_hybrids_TPM_comparison.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"Comparison complete: {len(comparison_df)} genes with TPM data in both datasets")
print(f"Pearson correlation (log10): r={pearson_r:.4f}, p={pearson_p:.2e}")
print(f"Spearman correlation: ρ={spearman_r:.4f}, p={spearman_p:.2e}")

# Correlation in RNA-seq between chondrocytes, osteochondral progenitors and cartilage organoids

In [ ]:
#Take colnames from Yamada table
Yamada_raw = pd.read_excel(YAMADA_RAW_XLSX, 
                     nrows = 1,
                     header=0)
steps_names = Yamada_raw.columns

# Take normalized TPM values from NCBI
Yamada_TPM = pd.read_csv(YAMADA_TPM_TSV,
                         sep = '\t',
                         header=0)
Yamada_TPM.columns = steps_names

# Convert gene IDs to gene names
GRCh38_annotation = pd.read_csv(NCBI_ANNOT_TSV_OLD,
                         sep = '\t',
                         header=0)

GRCh38_annotation = GRCh38_annotation[['GeneID','Symbol']]
Yamada_TPM = GRCh38_annotation.merge(
    Yamada_TPM,
    left_on="GeneID",   # column in Yamada_TPM
    right_on="Geneid",  # column in GRCh38_annotation
    how="right"          # or "inner", depending on what you want
)

#### Also add the articular cartilage chondrocytes from a different source:

In [ ]:
articular_cartilage_TPM = pd.read_csv(ARTICULAR_CARTILAGE_TPM_TSV, 
                     header=0,sep = '\t')

articular_cartilage_TPM = articular_cartilage_TPM.iloc[:,0:9]
articular_cartilage_TPM['articular_cartilage_mean'] = articular_cartilage_TPM.iloc[:, 1:].mean(axis=1)
articular_cartilage_TPM = articular_cartilage_TPM[['GeneID','articular_cartilage_mean']]


articular_cartilage_TPM = pd.merge(articular_cartilage_TPM, GRCh38_annotation, on='GeneID', how='outer') 
articular_cartilage_TPM = articular_cartilage_TPM.set_index('Symbol')
articular_cartilage_TPM = articular_cartilage_TPM[['articular_cartilage_mean']]
articular_cartilage_TPM = articular_cartilage_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates


In [ ]:
Yamada_TPM = Yamada_TPM[['Symbol',
                         'Day 0 (PSC) #1',
                         'Day 0 (PSC) #2',
                         'Day 2 (LPM) #1',
                         'Day 2 (LPM) #2',
                         'Day 4 (LBM) #1',
                         'Day 4 (LBM) #2',
                         'ExpLBM PRRX1 High#1',
                         'ExpLBM PRRX1 High#2',
                         #'ExpLBM PRRX1 Low#1',
                         #'ExpLBM PRRX1 Low#2',
                         'ExpLBM PRRX1 high 2DCI Step 2 #1',
                         'ExpLBM PRRX1 high 2DCI Step 2 #2', 'ExpLBM PRRX1 high 3DCI Step 1 #1',
                         'ExpLBM PRRX1 high 3DCI Step 1 #2', 'ExpLBM PRRX1 high 3DCI Step 2 #1',
                         'ExpLBM PRRX1 high 3DCI Step 2 #2',
                         'ExpLBM PRRX1 high 3DCI Step 3: 6wks #1',
                         'ExpLBM PRRX1 high 3DCI Step 3: 6wks #2']]

In [ ]:
rows_before = len(Yamada_TPM)

Yamada_TPM = Yamada_TPM.merge(
    articular_cartilage_TPM,
    left_on='Symbol',
    right_index=True,
    how='inner'
)

rows_after = len(Yamada_TPM)

print(f"Rows before merge: {rows_before:,}")
print(f"Rows after merge: {rows_after:,}")
print(f"Rows removed by inner join: {rows_before - rows_after:,}")

In [ ]:
# Modular scatter correlation plot for any two columns in Yamada_TPM
def plot_scatter_correlation(
    df,
    x_col,
    y_col,
    pseudocount=0.1,
    log_scale=True,
    alpha=0.5,
    s=12,
    color="#227C9D",
    title=None,
    save_path=None,
):
    plot_df = df[[x_col, y_col]].copy()
    plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
    plot_df[y_col] = pd.to_numeric(plot_df[y_col], errors="coerce")
    plot_df = plot_df.dropna()

    x = plot_df[x_col].values + pseudocount
    y = plot_df[y_col].values + pseudocount

    pearson_r, pearson_p = pearsonr(np.log10(x), np.log10(y)) if log_scale else pearsonr(x, y)
    spearman_r, spearman_p = spearmanr(x, y)

    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Calculate 2D density for coloring
    from matplotlib.colors import Normalize
    from scipy.stats import gaussian_kde
    
    # Calculate density on log-transformed data if applicable
    if log_scale:
        x_for_kde = np.log10(x)
        y_for_kde = np.log10(y)
    else:
        x_for_kde = x
        y_for_kde = y
    
    xy = np.vstack([x_for_kde, y_for_kde])
    kde = gaussian_kde(xy)
    density = kde(xy)
    
    # Cap density at 1
    #density = np.minimum(density, 1)
    
    # Normalize density to [0, 1] for colormap
    norm = Normalize(vmin=density.min(), vmax=density.max())
    colors = custom_cmap_bolder(norm(density))
    
    ax.scatter(x, y, alpha=alpha, s=s, c=colors, edgecolors="none")
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=custom_cmap_bolder, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Density', fontsize=FONT_SIZE_small - 2)

    if log_scale:
        ax.set_xscale("log")
        ax.set_yscale("log")

    min_val = min(x.min(), y.min())
    max_val = max(x.max(), y.max())
    ax.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1.2, alpha=0.6)

    ax.set_xlabel(x_col, fontsize=FONT_SIZE_small-2)
    ax.set_ylabel(y_col, fontsize=FONT_SIZE_small-2)
    #ax.set_title(title if title else f"{x_col} vs {y_col}", fontsize=FONT_SIZE_small)
    #ax.grid(True, alpha=0.25)

    # Format p-values, showing underflow when p-value is too small
    import sys
    pearson_p_str = f"p={pearson_p:.2e}" if pearson_p > 0 else f"p < {sys.float_info.min:.2e}"
    spearman_p_str = f"p={spearman_p:.2e}" if spearman_p > 0 else f"p < {sys.float_info.min:.2e}"
    
    textstr = (
        f"n = {len(plot_df):,}\n"
        f"Pearson's r = {pearson_r:.3f} ({pearson_p_str})\n"
        f"Spearman's ρ = {spearman_r:.3f} ({spearman_p_str})"
    )
    ax.text(
        0.03, 0.97, textstr,
        transform=ax.transAxes,
        va="top",
        fontsize=FONT_SIZE_small - 6,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
    )

    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.show()

    return {"n": len(plot_df), "pearson_r": pearson_r, "pearson_p": pearson_p, "spearman_r": spearman_r, "spearman_p": spearman_p}



In [ ]:
# Choose columns here (easy to change)
x_col = "ExpLBM PRRX1 High#1"
y_col = "ExpLBM PRRX1 high 3DCI Step 3: 6wks #1"

corr_stats = plot_scatter_correlation(
    Yamada_TPM,
    x_col=x_col,
    y_col=y_col,
    #title="Correlation between ExpLBM PRRX1 High#1 and 3DCI Step 3: 6wks #1",
    save_path=output_path + "ExpLBM_PRRX1_high_3DCI_Step_3_6wks_ExpLBM_PRRX1_High_1_scatter.png",
)

print(corr_stats)


In [ ]:
# Filter genes by distance from diagonal
# Uses existing DISTANCE_THRESHOLD variable (currently 0.1)
DISTANCE_THRESHOLD = 0.2 # log units (0.5 log units = ~3.16x fold difference)
# Prepare data
plot_df = Yamada_TPM[[x_col, y_col, "Symbol"]].copy()
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df[y_col] = pd.to_numeric(plot_df[y_col], errors="coerce")
plot_df = plot_df.dropna()

# Remove genes with TPM < 1 in BOTH datasets
before_filter_n = len(plot_df)
low_in_both_mask = (plot_df[x_col] < 1) & (plot_df[y_col] < 1)
removed_low_low_n = int(low_in_both_mask.sum())
plot_df = plot_df.loc[~low_in_both_mask].copy()
after_filter_n = len(plot_df)

print(f"Removed {removed_low_low_n:,} genes with TPM < 1 in both {x_col} and {y_col}.")
print(f"Remaining genes after TPM filter: {after_filter_n:,} (from {before_filter_n:,})")

# Values for plotting / distance (with pseudocount only for log safety)
x = plot_df[x_col].values + 0.1
y = plot_df[y_col].values + 0.1

# Calculate distance from diagonal in log space
x_log = np.log10(x)
y_log = np.log10(y)
distance_from_diag = np.abs(y_log - x_log) / np.sqrt(2)

# Create list of correlated genes (after TPM filtering)
correlated_genes = plot_df.loc[distance_from_diag <= DISTANCE_THRESHOLD, "Symbol"].tolist()
print(f"\nNumber of genes within {DISTANCE_THRESHOLD} log units from diagonal: {len(correlated_genes)}")
print(f"Example genes: {correlated_genes[:10]}")

# Create scatter plot colored by correlation status
fig, ax = plt.subplots(figsize=(8, 6))

correlated_set = set(correlated_genes)
colors = np.array(
    ["#2ecc71" if gene in correlated_set else "#95a5a6" for gene in plot_df["Symbol"].values]
)

ax.scatter(x, y, alpha=0.6, s=12, c=colors, edgecolors="none")

# Add diagonal line
min_val = min(x.min(), y.min())
max_val = max(x.max(), y.max())
ax.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1.2, alpha=0.6)

# Add distance band around diagonal
diag_x = np.array([min_val, max_val])
diag_y_upper = diag_x * (10 ** (DISTANCE_THRESHOLD * np.sqrt(2)))
diag_y_lower = diag_x / (10 ** (DISTANCE_THRESHOLD * np.sqrt(2)))
ax.fill_between(
    diag_x, diag_y_lower, diag_y_upper,
    alpha=0.1, color="green", label=f"±{DISTANCE_THRESHOLD} log units"
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(x_col, fontsize=FONT_SIZE_small)
ax.set_ylabel(y_col, fontsize=FONT_SIZE_small)
ax.set_title(
    "Genes by Correlation Status\n(Green: correlated, Gray: not correlated)",
    fontsize=FONT_SIZE_small
)

# Custom legend
legend_elements = [
    Patch(facecolor="#2ecc71", label=f"Correlated (n={len(correlated_genes)})"),
    Patch(facecolor="#95a5a6", label=f"Not correlated (n={len(plot_df)-len(correlated_genes)})"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=FONT_SIZE_small - 2)

plt.tight_layout()
plt.savefig(output_path + "Yamada_correlated_genes_annotation.png", bbox_inches="tight", dpi=300)
plt.show()

print("\nDistance threshold explanation:")
print(f"- {DISTANCE_THRESHOLD} log units = ~{10**(DISTANCE_THRESHOLD):.1f}x fold difference")

# Save correlated gene list (after TPM filtering)
correlated_genes_tsv = pd.DataFrame({"Symbol": correlated_genes})
correlated_genes_tsv.to_csv(
    output_path + "Yamada_correlated_genes_list.tsv",
    sep="\t",
    index=False
)

print(f"Saved {len(correlated_genes_tsv):,} genes to: {output_path}Yamada_correlated_genes_list.tsv")


In [ ]:
# Create Spearman correlation grid plot (log-transformed data)

# Prepare data - exclude 'Symbol' column and drop NAs
cols_for_corr = Yamada_TPM.columns[1:]  # Exclude 'Symbol'
corr_data = Yamada_TPM[cols_for_corr].apply(pd.to_numeric, errors='coerce').dropna()

# Log-transform the data (add pseudocount to avoid log(0))
corr_data_log = np.log10(corr_data + 0.1)

# Calculate Spearman correlation matrix on log-transformed data
spearman_corr_matrix = corr_data_log.corr(method='spearman')

# Plot Spearman correlation
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(spearman_corr_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='RdBu_r', 
            center=0, 
            vmin=-1, 
            vmax=1,
            cbar_kws={'label': 'Spearman ρ'},
            ax=ax,
            square=True,
            annot_kws={'fontsize': 9})
ax.set_title('Spearman Correlation - All Sample Combinations', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(output_path + "Yamada_correlation_grid_spearman.png", bbox_inches='tight', dpi=300)
plt.show()

print("Spearman Correlation Matrix:")
print(spearman_corr_matrix)

In [ ]:
# Create Pearson correlation grid plot (log-transformed data)
# Calculate Pearson correlation matrix on log-transformed data
pearson_corr_matrix = corr_data_log.corr(method='pearson')

# Plot Pearson correlation
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pearson_corr_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='RdBu_r', 
            center=0, 
            vmin=-1, 
            vmax=1,
            cbar_kws={'label': 'Pearson r'},
            ax=ax,
            square=True,
            annot_kws={'fontsize': 9})
ax.set_title('Pearson Correlation (log10-transformed) - All Sample Combinations', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(output_path + "Yamada_correlation_grid_pearson_log.png", bbox_inches='tight', dpi=300)
plt.show()

print("Pearson Correlation Matrix:")
print(pearson_corr_matrix)

# Human-Gorilla hybrids - Number of Genes

In [ ]:
hybrids_path = HYBRIDS_HG_POLARIZATION_TSV
hybrids_df = pd.read_csv(hybrids_path, sep="\t")


In [ ]:
hybrids_path_chimp = HYBRIDS_HC_POLARIZATION_TSV
hybrids_df_chimp = pd.read_csv(hybrids_path_chimp, sep="\t")